In [ ]:
# ============================================================
# PHASE 1: ALOS PALSAR-1 (L-BAND) EXTRACTION
# Target: Detect "Hidden Water" under dense reed canopy
# Years: 2007 - 2010
# ============================================================

import ee
import math

# !! Update with your actual GEE project ID !!
ee.Initialize(project='YOUR_GEE_PROJECT')

ASSET_PATH   = 'projects/YOUR_GEE_PROJECT/assets/'
DRIVE_FOLDER = 'FLATS_LakeNeusiedl_Results'

print("Initializing ALOS-1 L-Band extraction...")

# ============================================================
# 1. DEFINE STUDY AREA (Full Lake)
# ============================================================

# Austrian Side
reed_mask     = ee.Image(ASSET_PATH + 'reed_belt_mask_30m')
austrian_area = reed_mask.geometry().bounds()

# Hungarian Side
hungarian_reed_belt = ee.Geometry.Polygon([[
    [16.66224319458008, 47.74282272685786],
    [16.668079681396485, 47.68345515194875],
    [16.689022369384766, 47.63743924178832],
    [16.73708755493164,  47.63281228417411],
    [16.778972930908203, 47.63813325009341],
    [16.812618560791016, 47.64900484338546],
    [16.840427703857422, 47.66288018538474],
    [16.828068084716797, 47.68276174586255],
    [16.748417205810547, 47.68183719007681],
    [16.721294708251953, 47.72250214442143],
    [16.720951385498047, 47.7352034376655 ]
]])

full_lake = austrian_area.union(hungarian_reed_belt, maxError=1)

In [ ]:
# ============================================================
# 2. LOAD & CONVERT RADAR DATA (DN to Decibels)
# ============================================================

# The ALOS imagery stores data as Digital Numbers (DN). 
# Physics requires us to convert this to Backscatter Coefficient (gamma-naught in dB).
# The official JAXA conversion formula is: 10 * log10(DN^2) - 83.0

def convert_to_dB(image):
    # Select the HH polarization band (L-band horizontal transmit/horizontal receive)
    # This is the band that triggers "double-bounce" on flooded vegetation
    hh_dn = image.select('HH')
    
    # Apply JAXA Formula: dB = 10 * log10(DN^2) - 83.0
    hh_db = (hh_dn.pow(2)
             .log10()
             .multiply(10)
             .subtract(83.0)
             .rename('HH_dB'))
    
    # Apply a 3x3 low-pass filter to reduce radar "speckle" (salt-and-pepper noise)
    hh_filtered = hh_db.focal_median(radius=1, units='pixels')
    
    return hh_filtered.set('system:time_start', image.get('system:time_start'))

# Load the ALOS PALSAR Yearly Mosaics for our 4 target years
years = [2007, 2008, 2009, 2010]
alos_tasks = []

print("\nPreparing ALOS HH (L-Band) exports...")

for year in years:
    # Filter the collection for the specific year
    alos_col = (ee.ImageCollection('JAXA/ALOS/PALSAR/YEARLY/SAR')
                .filterDate(f'{year}-01-01', f'{year}-12-31'))
    
    # Get the single annual mosaic and convert to dB
    annual_alos_db = convert_to_dB(alos_col.first()).clip(full_lake)
    
    # Create an export task
    filename = f'ALOS_LBand_HH_dB_{year}'
    task = ee.batch.Export.image.toDrive(
        image           = annual_alos_db,
        description     = filename,
        folder          = DRIVE_FOLDER,
        fileNamePrefix  = filename,
        region          = full_lake,
        scale           = 25,          # ALOS native resolution
        crs             = 'EPSG:4326',
        maxPixels       = 1e10
    )
    task.start()
    alos_tasks.append(task)
    print(f"  Started export: {filename}")

print("\n✅ Phase 1 Complete! 4 radar tasks sent to Google Earth Engine.")
print("Check your Google Drive in a few minutes for the 4 '.tif' files.")